# pose-reregistration

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.data.annotated import load_label_map
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.satellite import GeoImage
from uavloc.features.segmentation import CLASSES, SegmentationFeatures, label_probabilities, load_unet
from uavloc.geo import LatLon, footprint_m, haversine_m, offset_latlon, rotate_to_north_up
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H, class_ncc_maps
from uavloc.plotting.correlation import class_overlay
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
SLUG = "pose-reregistration"
DEVICE = "mps"
SEED = 42
FRAME_LIMIT = None                # None = all 44; a small number for a quick look

YAW_RANGE_DEG = 45                # heading offsets searched: -45 .. +45
COARSE_STEP_DEG = 3
FINE_SPAN_DEG = 3                 # refine +-3 deg around the coarse best, 1 deg steps
ALTITUDE_SCALES = [round(0.5 + 0.05 * i, 2) for i in range(13)]   # 0.5 .. 1.1; archive calibration: AGL = logged - 13 m
SEARCH_RADIUS_M = 30.0
PEAK_FRACTION = 0.95              # cells scoring >= 95% of the peak count as "equally good" (ambiguity)
BLOCK_GAP_MS = 5000

CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water (as 002.2)
FOV_W_DEG = 60.0
DRONE_SEG_GSD_M = 0.10
MATCH_GSD_M = 0.385
SEG_CHECKPOINT_DRONE = "pocs/village-seg-baseline/model-round3-bn-drone.pt"
LABELS = ("gonchary-2025", "ortho", "ua-ortho-annotated")
SAT_IMAGE = "../archive/uav_localization_2025/data/satelites/satellite_gonchary.jpg"
SAT_TOP_LEFT = (49.762965, 24.166472)
SAT_BOTTOM_RIGHT = (49.759291, 24.174046)
EXAMPLE_FRAME = "drone-1743863874000"             # block 2, the worst case in the 002 walkthrough
ROWS_PER_FIGURE = 11

# Data

In [ ]:
set_seed(SEED)
HERE = REPO_ROOT / "pocs" / SLUG
frames = load_frames("drone-2025-easy")[:FRAME_LIMIT]
satellite = GeoImage.load(REPO_ROOT / SAT_IMAGE, LatLon(*SAT_TOP_LEFT), LatLon(*SAT_BOTTOM_RIGHT))
labels = label_probabilities(load_label_map(*LABELS), MATCH_GSD_M)
len(frames)

# Model

In [ ]:
features = SegmentationFeatures.from_satellite_probabilities(load_unet(REPO_ROOT / SEG_CHECKPOINT_DRONE, DEVICE), labels,
                                                             "all", DRONE_SEG_GSD_M, MATCH_GSD_M, DEVICE)

# Search

In [ ]:
def frame_footprint_m(frame: DroneFrame, scale: float) -> tuple[float, float]:
    return footprint_m(frame.altitude_m * scale, FOV_W_DEG, CAMERA_ASPECT_W_OVER_H)

In [ ]:
def drone_map(frame: DroneFrame, scale: float) -> torch.Tensor:
    """Drone class probabilities `(C, h, w)` on the match grid, for a footprint scaled by `scale`."""
    return torch.from_numpy(features.drone_probabilities(frame.image(), *frame_footprint_m(frame, scale)))

In [ ]:
def map_window(frame: DroneFrame, yaw_offset_deg: float, drone: torch.Tensor) -> torch.Tensor:
    """Label probabilities around the logged position, heading-up at logged yaw + offset, SEARCH_RADIUS_M wider."""
    h_m, w_m = drone.shape[1] * MATCH_GSD_M, drone.shape[2] * MATCH_GSD_M
    patch = labels.crop_rotated(frame.position, frame.yaw_deg + yaw_offset_deg, w_m + 2 * SEARCH_RADIUS_M,
                                h_m + 2 * SEARCH_RADIUS_M, out_mpp=MATCH_GSD_M)
    return torch.from_numpy(patch.transpose(2, 0, 1).copy())

In [ ]:
def combined_ncc(drone: torch.Tensor, window: torch.Tensor) -> np.ndarray | None:
    """Weighted mean of per-class NCC over the classes present in the drone map (002.2's score, before z-scoring)."""
    maps = class_ncc_maps(drone[None], window[None], CLASS_WEIGHTS)
    if not maps:
        return None
    return sum(CLASS_WEIGHTS[c] * m for c, m in maps.items()) / sum(CLASS_WEIGHTS[c] for c in maps)

In [ ]:
def cell_offset_m(row: float, col: float, drone: torch.Tensor, window: torch.Tensor) -> tuple[float, float]:
    """Drone centre for a surface cell, `(right_m, down_m)` from the window centre."""
    return ((col + drone.shape[2] / 2 - window.shape[2] / 2) * MATCH_GSD_M,
            (row + drone.shape[1] / 2 - window.shape[1] / 2) * MATCH_GSD_M)

In [ ]:
def evaluate(frame: DroneFrame, drone: torch.Tensor, yaw_offset_deg: float, scale: float) -> dict | None:
    window = map_window(frame, yaw_offset_deg, drone)
    surface = combined_ncc(drone, window)
    if surface is None:
        return None
    row, col = np.unravel_index(np.argmax(surface), surface.shape)
    right_m, down_m = cell_offset_m(row, col, drone, window)
    centre = surface[int(round((window.shape[1] - drone.shape[1]) / 2)), int(round((window.shape[2] - drone.shape[2]) / 2))]
    return {"yaw_offset_deg": yaw_offset_deg, "scale": scale, "score": float(surface[row, col]),
            "right_m": right_m, "down_m": down_m, "score_at_centre": float(centre), "surface": surface}

In [ ]:
def sweep(frame: DroneFrame, drone: torch.Tensor, offsets: np.ndarray, scale: float) -> list[dict]:
    return [r for r in (evaluate(frame, drone, float(d), scale) for d in offsets) if r is not None]

In [ ]:
def register(frame: DroneFrame) -> dict | None:
    """Coarse heading sweep at the logged scale, then the scale, then a fine heading sweep."""
    drones = {k: drone_map(frame, k) for k in ALTITUDE_SCALES}
    coarse = sweep(frame, drones[1.0], np.arange(-YAW_RANGE_DEG, YAW_RANGE_DEG + 1, COARSE_STEP_DEG), 1.0)
    if not coarse:
        return None
    best = max(coarse, key=lambda r: r["score"])
    scales = [r for k in ALTITUDE_SCALES for r in sweep(frame, drones[k], np.array([best["yaw_offset_deg"]]), k)]
    k = max(scales, key=lambda r: r["score"])["scale"]
    fine = sweep(frame, drones[k], best["yaw_offset_deg"] + np.arange(-FINE_SPAN_DEG, FINE_SPAN_DEG + 1), k)
    logged = evaluate(frame, drones[1.0], 0.0, 1.0)
    return {"final": max(fine, key=lambda r: r["score"]), "coarse": coarse, "scales": scales, "logged": logged, "drones": drones}

In [ ]:
def corrected_pose(frame: DroneFrame, result: dict) -> tuple[LatLon, float]:
    yaw_deg = frame.yaw_deg + result["yaw_offset_deg"]
    dx_m, dy_m = rotate_to_north_up(result["right_m"], result["down_m"], yaw_deg)
    return offset_latlon(frame.position, dx_m, -dy_m), yaw_deg

In [ ]:
def peak_extent_m(surface: np.ndarray) -> float:
    """Length of the region scoring >= PEAK_FRACTION of the peak, along its long axis: how far the answer could slide."""
    rows, cols = np.nonzero(surface >= PEAK_FRACTION * surface.max())
    if len(rows) < 2:
        return 0.0
    spread = np.linalg.eigvalsh(np.cov(np.stack([rows, cols]))).max()
    return float(4 * np.sqrt(spread) * MATCH_GSD_M)

# Run: one frame

In [ ]:
example_frame = next(f for f in frames if f.name == EXAMPLE_FRAME)
example = register(example_frame)
{k: v for k, v in example["final"].items() if k != "surface"}

# Run: all frames

In [ ]:
def summarise(frame: DroneFrame, reg: dict | None) -> dict:
    row = {"frame": frame.name, "logged_yaw_deg": frame.yaw_deg, "altitude_m": frame.altitude_m}
    if reg is None:
        return {**row, "registered": False}
    final = reg["final"]
    position, yaw_deg = corrected_pose(frame, final)
    return {**row, "registered": True, "yaw_offset_deg": final["yaw_offset_deg"], "scale": final["scale"],
            "shift_m": haversine_m(frame.position, position), "corrected_lat": position.lat, "corrected_lon": position.lon,
            "corrected_yaw_deg": yaw_deg % 360, "score": final["score"], "score_logged": reg["logged"]["score_at_centre"],
            "ambiguity_m": peak_extent_m(final["surface"])}

In [ ]:
registrations = {f.name: register(f) for f in frames}
results = pd.DataFrame([summarise(f, registrations[f.name]) for f in frames])
stamps = results["frame"].str.split("-").str[1].astype(int)
results["block"] = (stamps.diff() > BLOCK_GAP_MS).cumsum()
results["t_s"] = (stamps - stamps.iloc[0]) / 1000
results.to_csv(HERE / "results.csv", index=False)
results.drop(columns=["corrected_lat", "corrected_lon"]).round(2)

In [ ]:
registered = results[results["registered"]]
summary = {
    "frames": len(results), "registered": len(registered),
    "median_abs_yaw_offset_deg": float(registered["yaw_offset_deg"].abs().median()),
    "share_abs_yaw_offset_over_10deg": float((registered["yaw_offset_deg"].abs() > 10).mean()),
    "median_shift_m": float(registered["shift_m"].median()), "median_scale": float(registered["scale"].median()),
    "median_score_logged": float(registered["score_logged"].median()), "median_score_corrected": float(registered["score"].median()),
    "median_ambiguity_m": float(registered["ambiguity_m"].median()),
}
(HERE / "results.json").write_text(json.dumps(summary, indent=1))
summary

# Plots: one frame, step by step

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
coarse = pd.DataFrame(example["coarse"])
axes[0].plot(coarse["yaw_offset_deg"], coarse["score"], "o-")
axes[0].axvline(0, color="k", ls=":", label="logged heading"); axes[0].axvline(example["final"]["yaw_offset_deg"], color="r", label="best")
axes[0].set_xlabel("heading offset added to the log, deg"); axes[0].set_ylabel("best match score (NCC)"); axes[0].legend()
axes[0].set_title("1. try every heading: how well can the drone map match your map?")
scales = pd.DataFrame(example["scales"])
axes[1].plot(scales["scale"], scales["score"], "o-"); axes[1].set_xlabel("footprint scale (1 = logged altitude)")
axes[1].set_ylabel("best match score (NCC)"); axes[1].set_title("2. at that heading, try footprint sizes")
plt.tight_layout(); plt.savefig(HERE / "figures" / "example-sweeps.png", dpi=90); plt.show()

In [ ]:
def pose_panels(frame: DroneFrame, reg: dict) -> list[tuple[np.ndarray, str]]:
    """Drone frame; orthophoto and class overlay at the logged pose and at the re-registered pose."""
    final, k = reg["final"], reg["final"]["scale"]
    position, yaw_deg = corrected_pose(frame, final)
    track, woody = CLASSES.index("track"), CLASSES.index("woody")
    drone_logged, drone_best = reg["drones"][1.0][None], reg["drones"][k][None]
    at_logged = features.satellite_patch(frame.position, frame.yaw_deg, *frame_footprint_m(frame, 1.0))
    at_best = features.satellite_patch(position, yaw_deg, *frame_footprint_m(frame, k))
    return [(np.asarray(frame.image()), "drone frame"),
            (satellite.crop_rotated(frame.position, frame.yaw_deg, *frame_footprint_m(frame, 1.0)), "orthophoto at LOGGED pose"),
            (satellite.crop_rotated(position, yaw_deg, *frame_footprint_m(frame, k)), "orthophoto at RE-REGISTERED pose"),
            (class_overlay(drone_logged, at_logged, track, woody), f"overlay, logged: NCC {reg['logged']['score_at_centre']:.2f}"),
            (class_overlay(drone_best, at_best, track, woody), f"overlay, re-registered: NCC {final['score']:.2f}")]

In [ ]:
def plot_pose_rows(names: list[str]) -> plt.Figure:
    fig, axes = plt.subplots(len(names), 5, figsize=(25, 2.9 * len(names)), squeeze=False)
    for row, name in zip(axes, names):
        frame, r = next(f for f in frames if f.name == name), results.set_index("frame").loc[name]
        for ax, (image, title) in zip(row, pose_panels(frame, registrations[name])):
            ax.imshow(image); ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
        row[0].set_ylabel(f"{name[-7:]}  block {r['block']}\nheading {r['yaw_offset_deg']:+.0f} deg, shift {r['shift_m']:.1f} m\n"
                          f"scale {r['scale']:.1f}, could slide {r['ambiguity_m']:.1f} m", fontsize=8)
    fig.tight_layout()
    return fig

In [ ]:
fig = plot_pose_rows([EXAMPLE_FRAME])
fig.savefig(HERE / "figures" / "example-before-after.png", dpi=90); plt.show()

# Plots: all frames

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ok = results[results["registered"]]
for block, g in ok.groupby("block"):
    axes[0].plot(g["t_s"], g["logged_yaw_deg"], "o-", label=f"block {block}")
    axes[1].plot(g["t_s"], g["yaw_offset_deg"], "o-")
axes[0].set_ylabel("logged heading, deg"); axes[0].legend(fontsize=8, ncol=5)
axes[1].axhline(0, color="k", lw=.5); axes[1].set_ylabel("heading correction, deg"); axes[1].set_xlabel("time since first frame, s")
axes[0].set_title("logged heading (top) and the correction the map asks for (bottom), per frame")
plt.tight_layout(); plt.savefig(HERE / "figures" / "heading-corrections.png", dpi=90); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
margin_m = 20.0
ax.imshow(satellite.resampled(MATCH_GSD_M).pixels)
grid = satellite.resampled(MATCH_GSD_M)
for _, r in ok.iterrows():
    x0, y0 = grid.to_px(frames[[f.name for f in frames].index(r["frame"])].position)
    x1, y1 = grid.to_px(LatLon(r["corrected_lat"], r["corrected_lon"]))
    ax.annotate("", (x1, y1), (x0, y0), arrowprops={"arrowstyle": "->", "color": "yellow", "lw": 1.2})
    ax.plot(x0, y0, "c.", ms=4)
xs, ys = zip(*(grid.to_px(f.position) for f in frames))
ax.set_xlim(min(xs) - margin_m / MATCH_GSD_M * 3, max(xs) + margin_m / MATCH_GSD_M * 3)
ax.set_ylim(max(ys) + margin_m / MATCH_GSD_M * 3, min(ys) - margin_m / MATCH_GSD_M * 3)
ax.set_title("logged position (cyan) -> re-registered position (arrow tip)")
plt.tight_layout(); plt.savefig(HERE / "figures" / "position-shifts.png", dpi=90); plt.show()

In [ ]:
names = list(ok["frame"])
for i in range(0, len(names), ROWS_PER_FIGURE):
    fig = plot_pose_rows(names[i:i + ROWS_PER_FIGURE])
    fig.savefig(HERE / "figures" / f"before-after-{i // ROWS_PER_FIGURE + 1}.png", dpi=40); plt.show()